In [1]:
# 1. Upgrade pip just in case
!pip install --upgrade pip

# 2. Install a stable version of Rasterio that has pre-built binaries
# This bypasses the "GDAL/gdal-config" error by using a "wheel" file instead of compiling source code.
!pip install rasterio==1.3.10

# 3. Now install Sedona (It will see rasterio is already there and skip the bad step)
!pip install apache-sedona==1.6.1

# 4. Java is likely fine ("Nothing to do" means it's already installed), but we run it to be safe
!sudo yum install -y java-1.8.0-openjdk-devel

Loaded plugins: dkms-build-requires, extras_suggestions, kernel-livepatch,
              : langpacks, priorities, update-motd, versionlock
amzn2-core                                               | 3.6 kB     00:00     
amzn2extra-docker                                        | 2.9 kB     00:00     
amzn2extra-kernel-5.10                                   | 3.0 kB     00:00     
amzn2extra-livepatch                                     | 2.9 kB     00:00     
amzn2extra-lustre                                        | 2.5 kB     00:00     
centos-extras                                            | 2.9 kB     00:00     
copr:copr.fedorainfracloud.org:vbatts:shadow-utils-newxi | 3.3 kB     00:00     
https://download.docker.com/linux/centos/2/x86_64/stable/repodata/repomd.xml: [Errno 14] HTTPS Error 404 - Not Found
Trying other mirror.
nvidia-container-toolkit/x86_64/signature                |  833 B     00:00     
nvidia-container-toolkit/x86_64/signature                | 2.1 kB     00:03

In [2]:
# --- CELL 1: SETUP & INCOME DATA ---
import time
import json
import os
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StructType, StructField, StringType
from project_setup import get_spark_session, BASE_S3 

CENSUS_PATH = f"{BASE_S3}/LA_Census_Blocks_2020.geojson"
INCOME_PATH = f"{BASE_S3}/LA_income_2021.csv"
CRIME_PATH  = f"{BASE_S3}/LA_Crime_Data/LA_Crime_Data_2020_2025.csv"

spark = get_spark_session("Query5_Debug", executors="2", cores="4", memory="8g")
sc = spark.sparkContext

print(" Loading Income Data...")
df_income_raw = spark.read.option("header", "true").option("delimiter", ";").csv(INCOME_PATH)
df_income = df_income_raw.withColumn(
    "Median_Income", 
    F.regexp_replace(F.col("Estimated Median Income"), "[$,]", "").cast(DoubleType())
).withColumnRenamed("Zip Code", "Zip_Code").select("Zip_Code", "Median_Income", "Community")

print(f" Income Data Loaded: {df_income.count()} rows")

Configuring Environment for 'Query5_Debug'...
   Resource Config: 2 Executors | 4 Cores | 8g RAM
   JAVA_HOME set to: /usr/lib/jvm/java-1.8.0-openjdk-1.8.0.472.b08-1.amzn2.0.1.x86_64/jre
:: loading settings :: url = jar:file:/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ec2-user/.ivy2/cache
The jars for the packages stored in: /home/ec2-user/.ivy2/jars
org.apache.sedona#sedona-spark-shaded-3.4_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dc6c2cb7-26e4-4151-baf6-144eb175a3bc;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-shaded-3.4_2.12;1.6.1 in central
	found org.datasyslab#geotools-wrapper;1.6.1-28.2 in central
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.1026 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 520ms :: artifacts dl 18ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.apache.sedona#sedona-spark-shaded-3.4_2.12;1.6.1 from c

25/12/15 20:36:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
                                                                                

25/12/15 20:36:40 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/15 20:36:40 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/15 20:36:40 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/15 20:36:40 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/15 20:36:40 WARN SimpleFunctionRegistry: The function st_envelope_aggr replaced a previously registered function.
25/12/15 20:36:40 WARN SimpleFunctionRegistry: The function st_intersection_aggr replaced a previously registered function.
25/12/15 20:36:40 WARN SimpleFunctionRegistry: The function st_union_aggr replaced a previously registered function.
   Sedona Context Active
 Loading Income Data...
25/12/15 20:36:41 WARN MetricsConfig: Cannot locate configuration: tried h

In [3]:
# ---  DOWNLOAD CENSUS FILE LOCALLY ---
print(" Downloading Census File from S3 to Local Disk...")
local_file = "LA_Census_Blocks_2020.geojson"

cli_path = CENSUS_PATH.replace("s3a://", "s3://")
exit_code = os.system(f"aws s3 cp {cli_path} {local_file}")

if exit_code == 0:
    print(f" File downloaded successfully: {local_file}")
else:
    print(" Failed to download file. Check S3 path or permissions.")

download: s3://initial-notebook-data-bucket-dblab-905418150721/project_data/LA_Census_Blocks_2020.geojson to ./LA_Census_Blocks_2020.geojson
 File downloaded successfully: LA_Census_Blocks_2020.geojson


In [4]:
# ---  ROBUST PYTHON READ ---
import json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col
from sedona.spark import SedonaContext
from pyspark.storagelevel import StorageLevel

print(" Step 4: Reading JSON with Python & Creating DataFrame Explicitly...")


try:
    spark = SedonaContext.create(spark)
except:
    pass

local_filename = "LA_Census_Blocks_2020.geojson"

with open(local_filename, 'r') as f:
    data = json.load(f)

features_data = []
for feat in data['features']:
    props = feat.get('properties', {})
    geom = feat.get('geometry', None)
    
    if geom:
        geom_str = json.dumps(geom) 
        
        comm = props.get('COMM', 'Unknown')
        try:
            pop = int(props.get('POP20', 0))
        except:
            pop = 0
            
        features_data.append((comm, pop, geom_str))

print(f"   ...Prepared {len(features_data)} records in Python. Creating DataFrame...")

schema = StructType([
    StructField("Community", StringType(), True),
    StructField("Population", IntegerType(), True),
    StructField("geom_json", StringType(), True)
])

df_census = spark.createDataFrame(features_data, schema) \
    .repartition(200) \
    .selectExpr(
        "Community", 
        "Population", 
        "ST_SimplifyPreserveTopology(ST_GeomFromGeoJSON(geom_json), 0.0001) as census_geom"
    ) \
    .filter("census_geom IS NOT NULL")

df_census.persist(StorageLevel.MEMORY_AND_DISK)
count_census = df_census.count()
print(f" Census Ready: {count_census} areas (Robust Fast Mode).")

 Step 4: Reading JSON with Python & Creating DataFrame Explicitly...
25/12/15 20:38:15 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/15 20:38:15 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/15 20:38:15 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/15 20:38:15 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/15 20:38:15 WARN SimpleFunctionRegistry: The function st_envelope_aggr replaced a previously registered function.
25/12/15 20:38:15 WARN SimpleFunctionRegistry: The function st_intersection_aggr replaced a previously registered function.
25/12/15 20:38:15 WARN SimpleFunctionRegistry: The function st_union_aggr replaced a previously registered function.
   ...Prepared 91626 records in Python. Creating DataF

[Stage 11:====================================================> (195 + 2) / 200]

 Census Ready: 91626 areas (Robust Fast Mode).


In [ ]:
# --- CELL 5: EXECUTE JOIN  ---
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

print(" Step 5: Loading Crimes (Slim) & Running Spatial Join...")

df_crime = spark.read.option("header", "true").csv(CRIME_PATH) \
    .select(
        F.col("LAT"),
        F.col("LON"),
        F.col("DATE OCC")
    ) \
    .filter("LAT IS NOT NULL AND LON IS NOT NULL AND LAT != 0 AND LON != 0") \
    .filter("YEAR(TO_TIMESTAMP(`DATE OCC`, 'yyyy MMM dd hh:mm:ss a')) IN (2020, 2021)") \
    .selectExpr("ST_Point(CAST(LON AS DOUBLE), CAST(LAT AS DOUBLE)) as crime_geom") \
    .repartition(1000) 

print("   ...Starting Spatial Join (Trusting Catalyst)...")

df_joined = df_crime.join(df_census, F.expr("ST_Contains(census_geom, crime_geom)"))

df_stats = df_joined.groupBy("Community").agg(F.count("*").alias("Total_Crimes")) \
    .join(df_census.groupBy("Community").agg(F.sum("Population").alias("Total_Pop")), "Community") \
    .withColumn("Crime_Rate", F.col("Total_Crimes") / F.col("Total_Pop")) \
    .join(df_income, "Community") \
    .select("Community", "Median_Income", "Crime_Rate")

df_stats.persist(StorageLevel.MEMORY_AND_DISK) 

print("   ...Calculation started. Please wait...")
count = df_stats.count()
print(f"Final Result Size: {count} rows")

if count > 0:
    pdf = df_stats.toPandas()
    print("\n--- Simple Statistics ---")
    print(f"Correlation (Overall): {pdf['Median_Income'].corr(pdf['Crime_Rate']):.4f}")
    print(pdf.head())

⏳ Step 5: Loading Crimes (Slim) & Running Spatial Join...
   [DEBUG] Checking Crime Data loading speed...
